# Replication: Imai and Ratkovic (2014)

**Covariate Balancing Propensity Score**  
*Journal of the Royal Statistical Society, Series B*, 76(1), 243–263.

This notebook replicates the two main empirical analyses from the paper:

- **Section 3.1** — Kang–Schafer (2007) simulation study (Table 1)
- **Section 3.2** — LaLonde (1986) evaluation bias analysis (Table 2)

The Kang–Schafer simulation compares CBPS against standard logistic regression under four model specification scenarios. The LaLonde analysis applies CBPS-based propensity score matching to the NSW–PSID evaluation bias problem.

In [1]:
import warnings
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.spatial.distance import cdist
from scipy.optimize import minimize

import cbps
from cbps.datasets import load_lalonde_psid_combined

## Section 3.1: Kang–Schafer Simulation (Table 1)

The Kang–Schafer (2007) design is a standard benchmark for propensity score methods. Latent covariates $X^*$ are standard normal; observed covariates $X$ are nonlinear transformations of $X^*$. Treatment assignment follows a logistic model in $X^*$, and the potential outcome $Y(1)$ is linear in $X^*$ with $\mathbb{E}[Y(1)] = 210$.

Four scenarios cross correct/incorrect specification of the propensity score model and the outcome model.

### Data Generating Process

In [2]:
def kang_schafer_dgp(n, rng):
    """Generate one draw from the Kang-Schafer (2007) DGP.

    Latent covariates X* ~ N(0, I_4). Observed covariates X are nonlinear
    transformations of X*. Treatment follows logit(pr) = -X1* + 0.5*X2*
    - 0.25*X3* - 0.1*X4*. Outcome: Y = 210 + 27.4*X1* + 13.7*(X2*+X3*+X4*) + eps.
    """
    X_star = rng.standard_normal((n, 4))

    X_obs = np.column_stack([
        np.exp(X_star[:, 0] / 2),
        X_star[:, 1] / (1 + np.exp(X_star[:, 0])) + 10,
        (X_star[:, 0] * X_star[:, 2] / 25 + 0.6) ** 3,
        (X_star[:, 0] + X_star[:, 3] + 20) ** 2,
    ])

    logit_ps = -X_star[:, 0] + 0.5 * X_star[:, 1] - 0.25 * X_star[:, 2] - 0.1 * X_star[:, 3]
    ps_true = expit(logit_ps)
    T = rng.binomial(1, ps_true)

    Y = 210 + 27.4 * X_star[:, 0] + 13.7 * (X_star[:, 1] + X_star[:, 2] + X_star[:, 3])
    Y += rng.standard_normal(n)

    return X_star, X_obs, T, Y, ps_true

### Estimators

Four estimators for $\mathbb{E}[Y(1)]$: Horvitz–Thompson (HT), Hájek (IPW), weighted least squares (WLS), and doubly robust (DR).

In [3]:
def horvitz_thompson(Y, T, w):
    """Horvitz-Thompson estimator for E[Y(1)].

    Paper formula: mu_HT = (1/n) * sum(T_i * Y_i / pi(X_i))
    """
    n = len(Y)
    return np.sum(T * Y * w) / n


def ipw_hajek(Y, T, w):
    """Hajek (normalized IPW) estimator for E[Y(1)]."""
    return np.sum(T * Y * w) / np.sum(T * w)


def wls_estimator(Y, T, X, w):
    """WLS estimator: fit Y ~ X among treated, predict population mean."""
    X_aug = np.column_stack([np.ones(X.shape[0]), X])
    mask = T == 1
    if mask.sum() < X_aug.shape[1]:
        return np.nan
    try:
        beta = np.linalg.lstsq(
            X_aug[mask] * np.sqrt(w[mask, None]),
            Y[mask] * np.sqrt(w[mask]), rcond=None
        )[0]
        return np.mean(X_aug @ beta)
    except np.linalg.LinAlgError:
        return np.nan


def doubly_robust(Y, T, X, w):
    """Doubly robust (augmented IPW) estimator for E[Y(1)]."""
    X_aug = np.column_stack([np.ones(X.shape[0]), X])
    mask = T == 1
    if mask.sum() < X_aug.shape[1]:
        return np.nan
    try:
        beta = np.linalg.lstsq(X_aug[mask], Y[mask], rcond=None)[0]
        mu_hat = X_aug @ beta
        n = len(Y)
        return np.mean(mu_hat) + np.sum(T * w * (Y - mu_hat)) / n
    except np.linalg.LinAlgError:
        return np.nan


def compute_weights_from_ps(ps, T):
    """ATE-style IPW weights: 1/ps for treated, 1/(1-ps) for control."""
    return np.where(T == 1, 1.0 / ps, 1.0 / (1.0 - ps))

### Propensity Score Estimation

In [4]:
def fit_logistic(X_design, T_vec):
    """Standard logistic regression via MLE."""
    X_aug = np.column_stack([np.ones(len(T_vec)), X_design])

    def neg_loglik(beta):
        xb = np.clip(X_aug @ beta, -30, 30)
        return -np.sum(T_vec * xb - np.log(1 + np.exp(xb)))

    res = minimize(neg_loglik, np.zeros(X_aug.shape[1]), method='BFGS')
    return np.clip(expit(X_aug @ res.x), 1e-6, 1 - 1e-6)

### Single Simulation Draw

In [5]:
def run_one_simulation(n, rng):
    """Run one Monte Carlo draw for the Kang-Schafer design.

    Returns a dict mapping (scenario, method, estimator) -> point estimate.
    """
    X_star, X_obs, T, Y, ps_true = kang_schafer_dgp(n, rng)
    results = {}

    scenarios = {
        'Both correct':    (X_star, X_star),
        'PS correct':      (X_star, X_obs),
        'Outcome correct': (X_obs,  X_star),
        'Both wrong':      (X_obs,  X_obs),
    }

    estimators = {
        'HT':  lambda Y, T, X, w: horvitz_thompson(Y, T, w),
        'IPW': lambda Y, T, X, w: ipw_hajek(Y, T, w),
        'WLS': lambda Y, T, X, w: wls_estimator(Y, T, X, w),
        'DR':  lambda Y, T, X, w: doubly_robust(Y, T, X, w),
    }

    for scenario_name, (X_ps, X_out) in scenarios.items():
        ps_glm = fit_logistic(X_ps, T)

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            try:
                fit_exact = cbps.CBPS(treatment=T, covariates=X_ps, att=0,
                                      method='exact', verbose=0)
                ps_cbps1 = np.clip(fit_exact.fitted_values, 1e-6, 1 - 1e-6)
            except Exception:
                ps_cbps1 = ps_glm

            try:
                fit_over = cbps.CBPS(treatment=T, covariates=X_ps, att=0,
                                     method='over', verbose=0)
                ps_cbps2 = np.clip(fit_over.fitted_values, 1e-6, 1 - 1e-6)
            except Exception:
                ps_cbps2 = ps_glm

        method_ps = {
            'GLM':   ps_glm,
            'CBPS1': ps_cbps1,
            'CBPS2': ps_cbps2,
            'True':  np.clip(ps_true, 1e-6, 1 - 1e-6),
        }

        for mname, ps_vec in method_ps.items():
            w = compute_weights_from_ps(ps_vec, T)
            for ename, efunc in estimators.items():
                results[(scenario_name, mname, ename)] = efunc(Y, T, X_out, w)

    return results

### Run Simulation and Compile Results

In [6]:
def run_kang_schafer_simulation(n=200, n_sim=200, seed=2014):
    """Replicate Table 1 of Imai and Ratkovic (2014).

    Parameters
    ----------
    n : int
        Sample size per draw (paper uses 200 and 1000).
    n_sim : int
        Number of Monte Carlo replications (paper uses 10000).
    seed : int
        Random seed.
    """
    rng = np.random.default_rng(seed)
    true_mean = 210.0

    scenarios = ['Both correct', 'PS correct', 'Outcome correct', 'Both wrong']
    methods = ['GLM', 'CBPS1', 'CBPS2', 'True']
    estimator_names = ['HT', 'IPW', 'WLS', 'DR']

    all_results = {(s, m, e): [] for s in scenarios for m in methods for e in estimator_names}

    for i in range(n_sim):
        sim = run_one_simulation(n, rng)
        for key, val in sim.items():
            all_results[key].append(val)

    rows = []
    for scenario in scenarios:
        for method in methods:
            row = {'Scenario': scenario, 'PS Method': method}
            for est in estimator_names:
                vals = np.array(all_results[(scenario, method, est)])
                vals = vals[np.isfinite(vals)]
                if len(vals) > 0:
                    row[f'{est}_Bias'] = np.mean(vals) - true_mean
                    row[f'{est}_RMSE'] = np.sqrt(np.mean((vals - true_mean) ** 2))
                else:
                    row[f'{est}_Bias'] = np.nan
                    row[f'{est}_RMSE'] = np.nan
            rows.append(row)

    return pd.DataFrame(rows)

### Display Function

In [7]:
def print_table1(df, n, n_sim):
    """Format and print Table 1."""
    print(f'Table 1: Kang-Schafer Simulation (n={n}, {n_sim} replications)')
    print(f'True E[Y(1)] = 210\n')

    scenarios = ['Both correct', 'PS correct', 'Outcome correct', 'Both wrong']
    methods = ['GLM', 'CBPS1', 'CBPS2', 'True']
    estimators = ['HT', 'IPW', 'WLS', 'DR']

    header = f"{'Scenario':<20s} {'Method':<8s}"
    for est in estimators:
        header += f"  {est + '_Bias':>10s} {est + '_RMSE':>10s}"
    print(header)
    print('-' * len(header))

    for scenario in scenarios:
        for method in methods:
            row = df[(df['Scenario'] == scenario) & (df['PS Method'] == method)]
            if row.empty:
                continue
            line = f'{scenario:<20s} {method:<8s}'
            for est in estimators:
                bias = row[f'{est}_Bias'].values[0]
                rmse = row[f'{est}_RMSE'].values[0]
                line += f'  {bias:>10.2f} {rmse:>10.2f}'
            print(line)
        print()

### Table 1: n = 200

In [8]:
n_sim = 200  # Paper uses 10000; reduced for computational feasibility

print('Running Kang-Schafer simulation (n=200)...')
df_t1_200 = run_kang_schafer_simulation(n=200, n_sim=n_sim, seed=2014)
print_table1(df_t1_200, n=200, n_sim=n_sim)

Running Kang-Schafer simulation (n=200)...
Table 1: Kang-Schafer Simulation (n=200, 200 replications)
True E[Y(1)] = 210

Scenario             Method       HT_Bias    HT_RMSE    IPW_Bias   IPW_RMSE    WLS_Bias   WLS_RMSE     DR_Bias    DR_RMSE
-------------------------------------------------------------------------------------------------------------------------
Both correct         GLM             0.11       4.34        0.11       4.34        0.11       2.46        0.11       2.46
Both correct         CBPS1           0.21       3.37        0.21       3.37        0.11       2.46        0.11       2.46
Both correct         CBPS2          -1.56       3.64       -1.56       3.64        0.11       2.46        0.11       2.46
Both correct         True            0.02       4.51        0.02       4.51        0.11       2.46        0.11       2.46

PS correct           GLM             0.11       4.34        0.11       4.34        0.32       2.90        0.32       3.26
PS correct           CB

### Table 1: n = 1000

In [9]:
print('Running Kang-Schafer simulation (n=1000)...')
df_t1_1000 = run_kang_schafer_simulation(n=1000, n_sim=n_sim, seed=2014)
print_table1(df_t1_1000, n=1000, n_sim=n_sim)

Running Kang-Schafer simulation (n=1000)...
Table 1: Kang-Schafer Simulation (n=1000, 200 replications)
True E[Y(1)] = 210

Scenario             Method       HT_Bias    HT_RMSE    IPW_Bias   IPW_RMSE    WLS_Bias   WLS_RMSE     DR_Bias    DR_RMSE
-------------------------------------------------------------------------------------------------------------------------
Both correct         GLM             0.18       1.69        0.18       1.69        0.07       1.17        0.07       1.17
Both correct         CBPS1           0.17       1.39        0.17       1.39        0.07       1.17        0.07       1.17
Both correct         CBPS2          -0.59       1.57       -0.59       1.57        0.07       1.17        0.07       1.17
Both correct         True            0.05       2.27        0.05       2.27        0.07       1.17        0.07       1.17

PS correct           GLM             0.18       1.69        0.18       1.69        0.18       1.33        0.12       1.49
PS correct           

## Section 3.2: LaLonde Evaluation Bias Analysis (Table 2)

The evaluation bias test compares NSW experimental controls with PSID observational controls. Both groups are untreated, so the true evaluation bias is zero. Propensity scores are estimated under three model specifications (Linear, Quadratic, Smith–Todd) and three PS methods (GLM, CBPS-exact, CBPS-over), followed by 1-to-1 nearest neighbor matching.

In [10]:
def nearest_neighbor_match(ps_treated, ps_control, control_indices):
    """One-to-one nearest neighbor matching on log-odds with replacement.

    Paper (p. 256): 'matching is done on the log-odds of the estimated
    propensity score.'
    """
    logit_t = np.log(ps_treated / (1 - ps_treated)).reshape(-1, 1)
    logit_c = np.log(ps_control / (1 - ps_control)).reshape(-1, 1)
    dist = cdist(logit_t, logit_c, metric='euclidean')
    best = np.argmin(dist, axis=1)
    return control_indices[best]

In [11]:
def run_lalonde_analysis(seed=2014):
    """Replicate Table 2 of Imai and Ratkovic (2014).

    Uses the full LaLonde dataset (LaLonde.csv) which includes re74 for
    all observations. The 'exper' column distinguishes NSW experimental
    (exper=1, 722 obs) from PSID comparison (exper=0, 2490 obs).
    """
    from cbps.datasets.lalonde import _get_data_dir
    data_dir = _get_data_dir()
    full = pd.read_csv(data_dir / 'LaLonde.csv')

    # NSW experimental controls (exper=1, treat=0)
    nsw_ctrl = full[(full['exper'] == 1) & (full['treat'] == 0)].copy()
    # PSID comparison group (exper=0)
    psid = full[full['exper'] == 0].copy()

    nsw_ctrl_for_ps = nsw_ctrl.copy()
    nsw_ctrl_for_ps['select'] = 1
    psid_for_ps = psid.copy()
    psid_for_ps['select'] = 0

    combined = pd.concat([nsw_ctrl_for_ps, psid_for_ps], ignore_index=True)

    # Squared terms for quadratic and Smith-Todd specifications
    combined['age_sq'] = combined['age'] ** 2
    combined['educ_sq'] = combined['educ'] ** 2
    combined['re75_sq'] = combined['re75'] ** 2
    combined['re74_sq'] = combined['re74'] ** 2
    combined['hisp_re74zero'] = combined['hisp'] * (combined['re74'] == 0).astype(float)

    specs = {
        'Linear': 'select ~ age + educ + black + hisp + married + nodegr + re74 + re75',
        'Quadratic': ('select ~ age + educ + black + hisp + married + nodegr + re74 + re75'
                      ' + age_sq + educ_sq + re74_sq + re75_sq'),
        'Smith-Todd': ('select ~ age + educ + black + hisp + married + nodegr'
                       ' + re74 + re75 + age_sq + educ_sq + re74_sq + re75_sq'
                       ' + hisp_re74zero'),
    }

    ps_methods_cbps = {
        'CBPS1': {'method': 'exact', 'two_step': True},
        'CBPS2': {'method': 'over',  'two_step': True},
    }

    treated_mask = combined['select'] == 1
    control_mask = combined['select'] == 0
    treated_idx = np.where(treated_mask)[0]
    control_idx = np.where(control_mask)[0]

    def fit_glm_formula(formula_str, data):
        """Fit standard logistic regression (MLE) matching R's glm().

        Uses statsmodels GLM with IRLS algorithm, which matches R's
        glm(family=binomial) behavior including handling of
        quasi-complete separation.
        """
        import statsmodels.api as sm
        from statsmodels.genmod.families import Binomial
        dep, indep = formula_str.split('~')
        covars = [v.strip() for v in indep.split('+')]
        X_mat = data[covars].values.astype(float)
        y_vec = data[dep.strip()].values.astype(float)
        X_aug = np.column_stack([np.ones(len(y_vec)), X_mat])
        model = sm.GLM(y_vec, X_aug, family=Binomial())
        glm_fit = model.fit(maxiter=25)
        ps = glm_fit.fittedvalues
        return np.clip(ps, 1e-6, 1 - 1e-6)

    rows = []
    for spec_name, formula in specs.items():
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            try:
                ps_glm = fit_glm_formula(formula, combined)
            except Exception as e:
                print(f'  [{spec_name}/GLM] failed: {e}')
                ps_glm = None

        all_methods = {}
        if ps_glm is not None:
            all_methods['GLM'] = ps_glm

        for mname, mkwargs in ps_methods_cbps.items():
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                try:
                    fit = cbps.CBPS(formula=formula, data=combined,
                                    att=0, verbose=0, **mkwargs)
                    all_methods[mname] = np.clip(fit.fitted_values, 1e-6, 1 - 1e-6)
                except Exception as e:
                    print(f'  [{spec_name}/{mname}] failed: {e}')

        for mname in ['GLM', 'CBPS1', 'CBPS2']:
            if mname not in all_methods:
                rows.append({'Specification': spec_name, 'Method': mname,
                             'Eval Bias': np.nan, 'N Matched': 0})
                continue
            ps_all = all_methods[mname]
            matched_ctrl_idx = nearest_neighbor_match(
                ps_all[treated_idx], ps_all[control_idx], control_idx
            )
            re78_treated = combined.loc[treated_idx, 're78'].values
            re78_matched = combined.loc[matched_ctrl_idx, 're78'].values
            rows.append({
                'Specification': spec_name, 'Method': mname,
                'Eval Bias': np.mean(re78_treated) - np.mean(re78_matched),
                'N Matched': len(np.unique(matched_ctrl_idx)),
            })

    return pd.DataFrame(rows)

In [12]:
def print_table2(df):
    """Format and print Table 2."""
    print('Table 2: LaLonde Evaluation Bias (NSW Controls vs. PSID)')
    print('True evaluation bias = 0 (both groups are untreated)\n')

    header = f"{'Specification':<16s} {'Method':<8s} {'Eval Bias':>12s} {'N Matched':>10s}"
    print(header)
    print('-' * len(header))

    for _, row in df.iterrows():
        bias_str = f"{row['Eval Bias']:>12.1f}" if np.isfinite(row['Eval Bias']) else f"{'NA':>12s}"
        print(f"{row['Specification']:<16s} {row['Method']:<8s} {bias_str} {row['N Matched']:>10d}")

### Table 2: Results

In [13]:
print('Running LaLonde evaluation bias analysis...')
df_table2 = run_lalonde_analysis(seed=2014)
print_table2(df_table2)

Running LaLonde evaluation bias analysis...
Table 2: LaLonde Evaluation Bias (NSW Controls vs. PSID)
True evaluation bias = 0 (both groups are untreated)

Specification    Method      Eval Bias  N Matched
-------------------------------------------------
Linear           GLM            -756.5        130
Linear           CBPS1         -2834.9        152
Linear           CBPS2         -1220.0        133
Quadratic        GLM            5090.0          2
Quadratic        CBPS1          5090.0          2
Quadratic        CBPS2          5090.0          2
Smith-Todd       GLM           -1416.9        125
Smith-Todd       CBPS1         -1494.4        133
Smith-Todd       CBPS2          -485.0        122


## API Features: Summary and Diagnostics

The `cbps` package provides a statsmodels-style API. All result classes return dedicated summary objects from `summary()`, and `CBPSResults` now includes built-in diagnostics (convergence status, weight distribution, effective sample size).

In [ ]:
from cbps import CBPS, balance
from cbps.datasets import load_lalonde

data = load_lalonde()
fit = CBPS(
    formula='treat ~ age + educ + black + hisp + married + nodegr + re74 + re75',
    data=data, att=0, method='over'
)

# summary() returns CBPSSummary object with diagnostics block
print(fit.summary())

# balance() returns DataFrames with covariate names as row index
bal = balance(fit)
print('\nCovariate Balance (weighted):')
print(bal['balanced'])